# 云边联邦训练：风险估计可靠性
## Cloud-edge federated RL: reliability of risk estimates

唯一全局调参入口。先完成 `core.py` 的研究者算法，再启用真实训练与攻击。当前默认只执行版本核验和真实 Adult 数据准备，不生成模拟训练结果。

**问题**：调度诱导的训练分布变化是否造成风险低估？不确定性和有限预算审计能否改善决策？

工程说明见 README；任务顺序与研究约束见 docs。客户端编号在代码中从 0 开始。

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from cloud_edge_rl.notebook import execute_job
from cloud_edge_rl.pipeline import runtime_info

PROJECT_ROOT = Path.cwd()
RESULTS = {}
display(runtime_info())

## 1. 集中配置 / Central configuration

全部实验数值、变体、种子、阶段开关、输入产物与云资源只在下一单元修改。模型/算法参数是供您实现核心时使用的起点，不是已验证最优值。资源功率为模拟假设，非硬件实测。

20 美元是预实验目标，不是账单硬限额。首次保持单 L4、串行任务。`estimated_seconds_by_batch` 由 benchmark 阶段测量后从产物读取，不填虚构攻击耗时。恢复或重跑请改 `experiment.tag`，通过 inputs 显式引用已有 checkpoint；同名失败结果不会被覆盖。

In [ ]:
# 唯一调参区 / The ONLY place to edit experiment values.
# 默认只执行本地版本检查和真实数据准备；学习与攻击核心仍由您填写。
CONFIG = {
    "experiment": {"tag": "pilot-001", "variant": "setup"},
    "data": {
        "url": "https://archive.ics.uci.edu/static/public/2/adult.zip",
        "validation_fraction": 0.2,
        "split_seed": 42,
        "partition_seed": 73,
        "num_clients": 3,
        "partition": "iid",
        "label_concentration": 0.5,
    },
    "model": {
        "hidden_sizes": [64, 64],
        "activation": "relu",
        "output_size": 1,
        "loss": "binary_cross_entropy_with_logits",
        "learning_rate": 0.01,
        "optimizer": "sgd",
        "device": "cuda",
        "dtype": "float32",
    },
    "environment": {
        "client_sets": [[0, 1], [0, 2], [1, 2], [0, 1, 2]],
        "batch_sizes": [8, 32, 64],
        "max_rounds": 30,
        "time_budget_seconds": 120.0,
        "sample_budget": 3000,
        "target_validation_loss": 0.3,
        "utility_prior": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        "utility_ema_decay": 0.8,
    },
    "resources": {
        "interval_seconds": 2.0,
        "bandwidth_mbps": [5.0, 10.0, 20.0],
        "download_mbps": [20.0, 40.0, 60.0],
        "samples_per_second": [400.0, 800.0, 1200.0],
        "compute_watts": [5.0, 8.0, 12.0],
        "transmit_watts": [1.0, 1.5, 2.0],
        "receive_watts": [0.5, 0.7, 1.0],
        "bandwidth_jitter": 0.3,
        "load_range": [0.0, 0.6],
        "aggregation_seconds": 0.005,
        "server_watts": 30.0,
    },
    "reward": {
        "time_weight": 0.05,
        "energy_weight": 0.01,
        "risk_weight": 0.0,
        "uncertainty_weight": 0.0,
        "time_scale": 1.0,
        "energy_scale": 10.0,
    },
    "baseline": {"fixed_action": 4, "round_robin_actions": [1, 4, 7]},
    "risk": {
        "mode": "unavailable",
        "ensemble_size": 5,
        "hidden_sizes": [32, 32],
        "learning_rate": 0.001,
        "epochs": 20,
        "batch_size": 64,
        "device": "cuda",
        "validation_group_fraction": 0.2,
    },
    "rl": {
        "algorithm": "dqn",
        "device": "cuda",
        "hidden_sizes": [128, 128],
        "total_steps": 3000,
        "learning_rate": 0.0003,
        "gamma": 0.99,
        "batch_size": 64,
        "buffer_size": 20000,
        "learning_starts": 256,
        "train_frequency": 1,
        "target_update_interval": 100,
        "epsilon_start": 1.0,
        "epsilon_end": 0.05,
        "epsilon_decay_steps": 2000,
        "checkpoint_interval": 500,
        "ppo": {
            "rollout_steps": 128,
            "epochs": 4,
            "clip_ratio": 0.2,
            "gae_lambda": 0.95,
            "entropy_weight": 0.01,
        },
    },
    "attack": {
        "method": "gradient_matching",
        "iterations": 200,
        "restarts": 1,
        "learning_rate": 0.1,
        "labels_known": True,
        "device": "cuda",
    },
    "audit": {
        "policy": "periodic",
        "max_calls": 6,
        "max_seconds": 600.0,
        "period": 5,
        "ranking_floor": 0.01,
        "numeric_tolerance": 0.05,
        "device": "cuda",
        "estimated_seconds_by_batch": None,
    },
    "execution": {
        "split": "train",
        "save_updates": True,
        "warmup_actions": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
    },
    "seeds": {
        "pilot": [7],
        "train": [101, 102, 103, 104, 105],
        "validation": [201, 202],
        "test": [301, 302, 303, 304, 305],
    },
    "modal": {
        "app_name": "cloud-edge-rl",
        "volume_name": "cloud-edge-rl-artifacts",
        "gpu": "L4",
        "cpu": 2.0,
        "memory_mb": 8192,
        "timeout_seconds": 1800,
        "max_containers": 1,
        "scaledown_window": 2,
        "budget_usd": 20.0,
        "detach": True,
    },
    "variants": {
        "setup": {
            "method": "fixed",
            "risk_mode": "unavailable",
            "risk_weight": 0.0,
            "uncertainty_weight": 0.0,
        },
        "round-robin": {
            "method": "round_robin",
            "risk_mode": "unavailable",
            "risk_weight": 0.0,
            "uncertainty_weight": 0.0,
        },
        "fixed": {
            "method": "fixed",
            "risk_mode": "fitted",
            "risk_weight": 0.1,
            "uncertainty_weight": 0.0,
        },
        "time-greedy": {
            "method": "time_greedy",
            "risk_mode": "fitted",
            "risk_weight": 0.1,
            "uncertainty_weight": 0.0,
        },
        "risk-greedy": {
            "method": "risk_greedy",
            "risk_mode": "fitted",
            "risk_weight": 0.1,
            "uncertainty_weight": 0.0,
        },
        "conservative-greedy": {
            "method": "risk_greedy",
            "risk_mode": "fitted",
            "risk_weight": 0.1,
            "uncertainty_weight": 1.0,
        },
        "dqn-no-privacy": {
            "method": "dqn",
            "risk_mode": "fitted",
            "risk_weight": 0.0,
            "uncertainty_weight": 0.0,
        },
        "dqn": {
            "method": "dqn",
            "risk_mode": "fitted",
            "risk_weight": 0.1,
            "uncertainty_weight": 0.0,
        },
        "conservative-dqn": {
            "method": "dqn",
            "risk_mode": "fitted",
            "risk_weight": 0.1,
            "uncertainty_weight": 1.0,
        },
        "ppo": {
            "method": "ppo",
            "risk_mode": "fitted",
            "risk_weight": 0.1,
            "uncertainty_weight": 0.0,
        },
    },
    "tuning": {
        "learning_rate": [0.0001, 0.0003, 0.001],
        "risk_weight": [0.03, 0.1, 0.3],
        "uncertainty_weight": [0.0, 0.5, 1.0],
        "trials": 6,
        "seed": 9001,
    },
    "analysis": {
        "run_ids": [],
        "audit_run_ids": [],
        "reference_method": "risk-greedy",
        "metric": "validation_loss",
        "x_axis": "elapsed_seconds",
        "risk_threshold": 0.5,
    },
    "workflow": {
        "reuse_completed": True,
        "jobs": {
            "runtime": {
                "enabled": True,
                "stage": "runtime_check",
                "backend": "local",
                "variants": ["setup"],
                "seed_group": "pilot",
                "risk_version": "unavailable",
                "split": "train",
                "inputs": {},
            },
            "data": {
                "enabled": True,
                "stage": "prepare_data",
                "backend": "local",
                "variants": ["setup"],
                "seed_group": "pilot",
                "risk_version": "unavailable",
                "split": "train",
                "inputs": {},
            },
            "modal_check": {
                "enabled": False,
                "stage": "runtime_check",
                "backend": "modal",
                "variants": ["setup"],
                "seed_group": "pilot",
                "risk_version": "unavailable",
                "split": "train",
                "inputs": {},
            },
            "collect": {
                "enabled": False,
                "stage": "collect",
                "backend": "modal",
                "variants": ["round-robin"],
                "seed_group": "pilot",
                "risk_version": "unavailable",
                "split": "train",
                "inputs": {},
            },
            "benchmark": {
                "enabled": False,
                "stage": "benchmark_attack",
                "backend": "modal",
                "variants": ["setup"],
                "seed_group": "pilot",
                "risk_version": "unavailable",
                "split": "train",
                "inputs": {"source_run_ids": []},
            },
            "audit": {
                "enabled": False,
                "stage": "audit",
                "backend": "modal",
                "variants": ["setup"],
                "seed_group": "pilot",
                "risk_version": "unavailable",
                "split": "train",
                "inputs": {"source_run_ids": [], "benchmark_run_id": None},
            },
            "fit_risk": {
                "enabled": False,
                "stage": "fit_risk",
                "backend": "modal",
                "variants": ["setup"],
                "seed_group": "pilot",
                "risk_version": "risk-v1",
                "split": "train",
                "inputs": {"audit_run_ids": []},
            },
            "train": {
                "enabled": False,
                "stage": "train_policy",
                "backend": "modal",
                "variants": ["dqn"],
                "seed_group": "pilot",
                "risk_version": "risk-v1",
                "split": "train",
                "inputs": {"risk_checkpoint": None, "previous_training_checkpoint": None},
            },
            "tune": {
                "enabled": False,
                "stage": "train_policy",
                "backend": "modal",
                "variants": ["dqn"],
                "seed_group": "pilot",
                "risk_version": "risk-v1",
                "split": "train",
                "inputs": {"risk_checkpoint": None, "previous_training_checkpoint": None},
            },
            "validate": {
                "enabled": False,
                "stage": "validate",
                "backend": "modal",
                "variants": ["dqn"],
                "seed_group": "validation",
                "risk_version": "risk-v1",
                "split": "validation",
                "inputs": {"training_run_ids": []},
            },
            "calibrate": {
                "enabled": False,
                "stage": "calibrate_risk",
                "backend": "modal",
                "variants": ["setup"],
                "seed_group": "pilot",
                "risk_version": "risk-v2",
                "split": "train",
                "inputs": {"audit_run_ids": []},
            },
            "evaluate": {
                "enabled": False,
                "stage": "evaluate",
                "backend": "modal",
                "variants": ["dqn"],
                "seed_group": "test",
                "risk_version": "risk-v1",
                "split": "test",
                "inputs": {"training_run_ids": []},
            },
            "evaluate_audits": {
                "enabled": False,
                "stage": "evaluate_audits",
                "backend": "modal",
                "variants": ["setup"],
                "seed_group": "test",
                "risk_version": "risk-v1",
                "split": "test",
                "inputs": {"source_run_ids": [], "benchmark_run_id": None},
            },
            "evaluate_baselines": {
                "enabled": False,
                "stage": "evaluate",
                "backend": "modal",
                "variants": ["fixed", "time-greedy", "risk-greedy", "conservative-greedy"],
                "seed_group": "test",
                "risk_version": "risk-v1",
                "split": "test",
                "inputs": {"risk_checkpoint": None},
            },
        },
    },
}

# 设备是运行资源选择；全模块使用同一显式选择。 / Share the selected compute device.
for section in ("attack", "audit", "risk", "rl"):
    CONFIG[section]["device"] = CONFIG["model"]["device"]

## 2. 框架核验与数据 / Runtime and data

保留官方测试集，仅训练部分拟合编码。首次下载真实 Adult ZIP 并记录 SHA256。缺失与未知类别分别编码；数据划分不依赖策略。

重复执行已完成阶段时读取原结果；改源码或配置会生成新的运行 ID。

In [ ]:
RESULTS["runtime"] = execute_job(CONFIG, "runtime", PROJECT_ROOT)
RESULTS["data"] = execute_job(CONFIG, "data", PROJECT_ROOT)
display(pd.DataFrame(RESULTS["data"]))

## 3. Modal 连通性 / Modal connectivity

仅在配置区将 `modal_check.enabled` 设为 True 才提交。首次会构建锁定镜像，再检查真实 CUDA 可用性，不运行学习核心。调用 ID 保存到 output/submissions；云端产物回收到本机 runs。

终端一次性认证使用 `uv run modal setup`；凭证不写入 Notebook 或 Git。

In [ ]:
RESULTS["modal_check"] = execute_job(CONFIG, "modal_check", PROJECT_ROOT)
display(RESULTS["modal_check"])

## 4. 真实预实验和风险采样 / Real pilot and risk collection

先实现 `FederatedBackend.reset/step_round/checkpoint/load_checkpoint`。初始采样用显式 risk=None 模式，risk_available 为 0，审计预测为 None，不表示真实风险等于零。固定的 12 动作预热计入时间、样本与暴露预算。需要小型真实攻击前先确保至少覆盖 8/32/64 三种 batch。

后续启用 train 时，同样在用户训练循环中调用 `baselines.run_warmup`，与基线使用相同历史信息与成本。

In [ ]:
RESULTS["collect"] = execute_job(CONFIG, "collect", PROJECT_ROOT)
display(RESULTS["collect"])

## 5. 攻击成本与离线标签 / Attack cost and offline labels

实现 `GradientAttack.reconstruct` 后，把 collect 的 run_id 填入配置区 benchmark.inputs.source_run_ids。benchmark 对每种 batch 的一个已上传梯度实测耗时；再把 benchmark run_id 填入 audit.inputs.benchmark_run_id，并填写 source_run_ids。

定期/随机/主动审计使用相同调用和算时预算。已启动的单次攻击可能超出剩余估算，超出量会完整记录；主动审计不能在没有真实风险模型时使用。

In [ ]:
RESULTS["benchmark"] = execute_job(CONFIG, "benchmark", PROJECT_ROOT)
RESULTS["audit"] = execute_job(CONFIG, "audit", PROJECT_ROOT)
display(RESULTS["benchmark"] + RESULTS["audit"])

## 6. 风险拟合 / Risk fitting

填写 fit_risk.inputs.audit_run_ids，然后实现 RiskEstimator 的 fit/predict/checkpoint/load_checkpoint。输入按完整 run_id 分组，验证组划分由您在核心中完成。任何 test/validation 审计标签都不能进入此拟合入口。

输出 runs/<run_id>/risk.pt，后续训练显式引用此路径和 risk_version。

In [ ]:
RESULTS["fit_risk"] = execute_job(CONFIG, "fit_risk", PROJECT_ROOT)
display(RESULTS["fit_risk"])

## 7. DQN 主线与调参 / DQN and tuning

实现 train_agent、Agent 和 load_agent。配置区 train.inputs.risk_checkpoint 指向已完成的风险版本。普通/保守/无隐私项 DQN 通过 variants 明确区分。PPO 保留同一接口，主线完成后再启用。

`tuning` 声明搜索空间；所有候选应使用相同验证种子与预算。默认只展示候选；在集中配置启用 tune 才逐候选运行相同预算训练。选择最优参数必须基于验证场景，最终 test 不参与选择。

将训练或 tune 产生的 run_id 填入 validate.inputs.training_run_ids 并启用 validate。框架从每个训练快照恢复对应超参数，在共同验证种子上评价，且不读取最终测试集。选定 trial 后在 evaluate.inputs.training_run_ids 引用其运行；框架保留其训练快照用于最终评价。 / Validation restores each trial snapshot on shared held-out seeds.

In [ ]:
from cloud_edge_rl.tuning import sample_trials

display(pd.DataFrame(sample_trials(CONFIG["tuning"])))
RESULTS["tune"] = execute_job(CONFIG, "tune", PROJECT_ROOT)
RESULTS["train"] = execute_job(CONFIG, "train", PROJECT_ROOT)
display(RESULTS["train"])
RESULTS["validate"] = execute_job(CONFIG, "validate", PROJECT_ROOT)
display(RESULTS["validate"])

## 8. 分阶段审计校准 / Stagewise calibration

将新采集的 train 审计运行加入 calibrate.inputs.audit_run_ids（包含原训练标签以免遗忘）。新版本在新阶段使用；阶段内始终冻结。用户 RL 核心需要保留策略权重、清空旧版本 replay；同版本恢复则还原优化器、RNG、replay 及 RecordingEnv.snapshot。

先在同一上传记录池比较三种审计规则，再比较闭环策略，避免把采样轨迹差异当作选点收益。

In [ ]:
RESULTS["calibrate"] = execute_job(CONFIG, "calibrate", PROJECT_ROOT)
display(RESULTS["calibrate"])

## 9. 独立最终评价 / Held-out evaluation

冻结算法、风险模型和全部调参选择；将选定 trial 的训练 run_id 填入 evaluate.inputs.training_run_ids；框架恢复其完整参数和两个 checkpoint，不用当前默认值覆盖获选设置。评估使用 test 种子，业务测试集只在回合结束后读取。上传记录标记 test；其攻击标签不能回流 fit。

将 evaluate run_id 填入 evaluate_audits.source_run_ids。独立面板可将 audit.policy 设为 panel，覆盖实际保存的动作/客户端；覆盖不足应补充独立轮询采集，不能伪造反事实梯度。

透明基线使用 evaluate_baselines 阶段，填写同一冻结风险检查点，并在验证后锁定 fixed_action。 / Transparent baselines use the same frozen risk checkpoint.

In [ ]:
RESULTS["evaluate_baselines"] = execute_job(CONFIG, "evaluate_baselines", PROJECT_ROOT)
RESULTS["evaluate"] = execute_job(CONFIG, "evaluate", PROJECT_ROOT)
RESULTS["evaluate_audits"] = execute_job(CONFIG, "evaluate_audits", PROJECT_ROOT)
display(RESULTS["evaluate"] + RESULTS["evaluate_audits"])

## 10. 结果汇总与课程材料 / Analysis and deliverables

在集中配置 analysis 中填写要比较的已完成 run_id。分别按模拟时间和累计处理量绘图；每个种子作为独立统计单位，展示平均值与离散程度。未发生风险漂移、校准无效或 DQN 与贪心接近都属于可报告结果。

报告 8–20 页（不含参考文献），Arial 12、1.5 倍行距、1 英寸页边距；展示 15 分钟。Git、复现记录与贡献说明保留完整。

In [ ]:
from cloud_edge_rl.analysis import (
    analyze_risk_runs,
    pairwise_summary,
    plot_learning_curves,
    plot_risk_calibration,
    summarize_runs,
)

if CONFIG["analysis"]["run_ids"]:
    table = summarize_runs(PROJECT_ROOT, CONFIG["analysis"]["run_ids"])
    display(table)
    display(
        pairwise_summary(
            table, CONFIG["analysis"]["metric"], CONFIG["analysis"]["reference_method"]
        )
    )
    plot_learning_curves(
        PROJECT_ROOT,
        CONFIG["analysis"]["run_ids"],
        PROJECT_ROOT / "output" / "figures",
        CONFIG["analysis"]["x_axis"],
    )
if CONFIG["analysis"]["audit_run_ids"]:
    display(
        analyze_risk_runs(
            PROJECT_ROOT, CONFIG["analysis"]["audit_run_ids"], CONFIG["analysis"]["risk_threshold"]
        )
    )
    plot_risk_calibration(
        PROJECT_ROOT, CONFIG["analysis"]["audit_run_ids"], PROJECT_ROOT / "output" / "figures"
    )